# Near-real-time IED detection from an EGI `.mff`

Interictal epileptiform discharge detection for sleep EEG, fast enough to drive
closed-loop stimulation. ~36 ms from the end of a 250 ms window to a decision:
25 ms of lookahead in the band-pass, ~11 ms of compute.

The detector is this notebook. It also needs `models/`, `constants.py` and the
packages in `requirements.txt`, and nothing else.

## The chain

```
input     EGI .mff, 1000 Hz, channels E227 and E254
stage 1   causal Butterworth ORDER 3 0.1 Hz high-pass              0 ms
          51-tap symmetric 70 Hz firwin, read 25 samples late     25 ms
stage 3   trailing 2 s box detrend                                 0 ms
stage 6   expanding z-score, seeded on the first 20 min            0 ms
model 1   the frozen notebook booster, 210 features   (never refit)
model 2   XGBoost reg:squarederror on those 210 features + the frozen logit,
          target logit(teacher) - logit(frozen)
score     sigmoid(logit(frozen) + alpha * correction)
stage 7   threshold 0.70, 2 s refractory, 20 min bins
output    (time_sec, probability) per detection
```

Model 2 is the contribution. It learns from past recordings how far the causal
score sits from what the same classifier would say with 16.5 s of future, and
adds that back. The offline chain only makes training labels; it is not here.

## How well it works

Agreement with an offline reference chain (33001 taps, 16.5 s of lookahead), not
clinical ground truth. No clinician annotated any of this. Scored exact epoch: a
detection counts only if it lands on the same 250 ms epoch.

Leave-one-recording-out over the three recordings, threshold 0.70:

| | precision | recall | F1 |
|---|---|---|---|
| frozen classifier alone | 0.316 | 0.366 | 0.339 |
| **+ the NREM-trained correction** | **0.388** | **0.392** | **0.383** |

Per recording: DH3 0.426, AO0 0.268, MF7 0.455. **Expect this, not better, on a
recording the model has not seen.** The correction costs 0.14 ms per decision.

## Before you rely on it

* Three recordings, all EGI 1000 Hz, all N3-heavy.
* **The shipped correction has no held-out patient.** Depth and alpha were
  selected on all three recordings. The honest estimate is the table above.
* Channels are hard-wired to E227/E254 at 1000 Hz.
* **Do not tune the threshold.** Four rules that move it were tried and all four
  lose. Precision above ~0.39 is not reachable by thresholding: it is flat from
  0.70 to 0.80 while recall collapses. `dev/RESULTS.md` §2 and §6.
* 20 min of warm-up per night is structural. Stage 6 cannot normalise a patient
  it has not seen.

**To run it:** fill in the parameters cell below, then Run All.

In [ ]:
# Colab only: uncomment, run, then Runtime -> Restart session (the pinned
# numpy/scipy need it).  Locally the venv in the README already has these.
# !pip install -r requirements.txt

---
# Parameters

**This is the only cell to edit.** Point it at a recording, set the two
switches, then Run All. Everything after this cell is the chain and the run.

In [ ]:
# --- the recording -----------------------------------------------------
MFF_DIR          = "path/to/RECORDING.mff"  # the .mff directory, not a file
START_MINUTE     = 0.0                      # first minute to analyse
END_MINUTE       = 480.0                    # last minute

# --- the detector ------------------------------------------------------
USE_CORRECTION   = True          # False = frozen classifier alone, the baseline
DROP_SHORT_FINAL = False         # False = live-device behaviour; see stage 7
OUTPUT_CSV       = "spikes.csv"  # None to print the detections instead

# Everything else lives in constants.py and is not a per-run parameter:
#   DETECTION_THRESHOLD    0.70    the correction was picked at this value
#   NORMALISATION_MINUTES  20 min  the seeding block; the z-score was fitted with it
#   FILTER_WARMUP_MINUTES  1 min   filter warm-up read before START_MINUTE

print(f"{MFF_DIR}\n{START_MINUTE:g}-{END_MINUTE:g} min, correction "
      f"{'on' if USE_CORRECTION else 'OFF'}")

In [ ]:
"""Imports, version check, and the one expected warning."""
import json
import os
import re
import struct
import sys
import time
import warnings

import numpy as np
import pandas as pd
import scipy.signal as sp_signal
import scipy.stats as sp_stats
import joblib
import xgboost as xgb

import mne
import mne.filter as _mne_filter
import mne_features.univariate

# An old pickle, so loading it warns.  Re-saving would change the numbers.
warnings.filterwarnings("ignore", message=".*serialized model.*")
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Pinned because `mne_features` definitions move between releases.  Other
# versions run fine, they just give different numbers.
PINNED_VERSIONS = {"numpy": "2.0.2", "pandas": "2.3.3", "scipy": "1.13.1",
                   "xgboost": "2.1.4", "mne": "1.8.0", "mne-features": "0.3"}
import scipy
import mne_features
_installed = {"numpy": np.__version__, "pandas": pd.__version__,
              "scipy": scipy.__version__, "xgboost": xgb.__version__,
              "mne": mne.__version__, "mne-features": mne_features.__version__}
_mismatched = {package: (version, PINNED_VERSIONS[package])
               for package, version in _installed.items()
               if version != PINNED_VERSIONS[package]}
print(f"python {sys.version.split()[0]}   " +
      "  ".join(f"{package} {version}" for package, version in _installed.items()))
if _mismatched:
    print("\nWARNING - versions differ from the pinned set: " +
          ", ".join(f"{package} {found} (pinned {pinned})"
                    for package, (found, pinned) in _mismatched.items()) +
          "\n          the detector will run, but the published numbers are "
          "not guaranteed.\n          pip install -r requirements.txt")

In [ ]:
"""Every constant the chain uses, from `constants.py`.

No bare number below this cell is a chain value; they all have names, defined
once in `constants.py`.  None of them is a tuning knob - change one and the
published numbers stop applying.
"""


def find_repo_root():
    """The directory holding `constants.py` and `models/`.

    Not the cwd: on Colab that is `/content`, not the repo, and the failure
    reads as `ModuleNotFoundError: constants` rather than a missing `%cd`.
    """
    def has_required_files(candidate):
        return (os.path.isfile(os.path.join(candidate, "constants.py"))
                and os.path.isfile(os.path.join(candidate, "models", "xgb_frozen.pkl")))

    start_dir = os.path.abspath(os.getcwd())
    candidates = [start_dir] + [
        os.path.abspath(os.path.join(start_dir, *[os.pardir] * levels_up))
        for levels_up in (1, 2)]
    for base_dir in ("/content", "/content/drive/MyDrive",
                     os.path.expanduser("~"), start_dir):
        if os.path.isdir(base_dir):
            try:
                candidates += [os.path.join(base_dir, entry)
                               for entry in sorted(os.listdir(base_dir))]
            except OSError:
                pass
    for candidate in candidates:
        if os.path.isdir(candidate) and has_required_files(candidate):
            return candidate
    raise SystemExit(
        "Could not find constants.py next to models/xgb_frozen.pkl.\n\n"
        "This notebook needs constants.py and models/ beside it.  On Colab,\n"
        "upload to /content:\n"
        "    constants.py\n"
        "    models/   xgb_frozen.pkl, correction.json, correction_meta.json\n"
        "    requirements.txt\n"
        "then run !pip install -r requirements.txt, restart the session (the\n"
        "pinned numpy/scipy need it) and start again from the top.\n\n"
        "Looked in:\n  " + "\n  ".join(dict.fromkeys(candidates[:12])))


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
from constants import *      # noqa: F401,F403  - see constants.py for all of it
import constants as CONSTANTS        # so you can look up where any value comes from

# paths are anchored to REPO_ROOT, so the working directory doesn't matter
MODELS_DIR = os.path.join(REPO_ROOT, MODEL_DIR)

print(f"repository   {REPO_ROOT}")
_n_constants = len([name for name in dir(CONSTANTS) if name.isupper()])
print(f"constants.py {_n_constants} constants")
print(f"{FIR_TAP_COUNT} taps -> {LOOKAHEAD_MS:.0f} ms of lookahead, "
      f"order-{HIGHPASS_ORDER} high-pass at {BANDPASS_LOW_HZ} Hz, "
      f"{DETREND_WINDOW_SECONDS:g} s trailing box")
print(f"threshold {DETECTION_THRESHOLD:.2f}, {REFRACTORY_SECONDS} s refractory "
      f"({REFRACTORY_SECONDS * EPOCHS_PER_SECOND} epochs), {BIN_LENGTH_MINUTES} min bins "
      f"({EPOCHS_PER_BIN} epochs)")
print(f"normalisation {NORMALISATION_MINUTES:g} min "
      f"({NORMALISATION_SECONDS:g} s), warm-up "
      f"{FILTER_WARMUP_MINUTES:g} min - neither is a default to lower")

---
## The seam: `SampleFeed`

The chain touches the recording through one interface:

```python
class SampleFeed:
    def take(self, n_samples): ...   # the next n samples.  The entire API.
```

Nothing returns a sample the wall clock has not reached, so "no lookahead beyond
25 ms" is checkable. The 25 ms is spent once, in the band-pass.

**This is the plug-in point for live zEEG.** Swap `MffFileFeed` for an
amplifier-backed feed and nothing else changes. Sketch at the bottom of the next
cell.

### What live operation still needs

This is a simulator over a file. The loop is bounded, `MffFileFeed` raises
`EOFError` at end of file, and one exception ends the run. The one that matters
is gap handling: there is none, so a dropped packet is treated as contiguous and
corrupts the filter state, the z-score and every later timestamp without raising
anything. One of the three recordings (DH3) already has a 4.6 min gap. What each
of these takes is in the README's "Next steps".

Two more are structural, not bugs: the 25 ms of lookahead is real delay on a
device, and so is the 20 min normalisation.


In [ ]:
class SampleFeed:
    """Samples in arrival order.  No method returns the future."""

    def take(self, n_samples):
        raise NotImplementedError

    io_sec = 0.0


class InMemoryFeed(SampleFeed):
    """A recording already in memory (a raw cache), handed out sequentially."""

    def __init__(self, data, start_sample):
        self._data = data
        self._next_sample = int(start_sample)

    def take(self, n_samples):
        if self._next_sample + n_samples > self._data.shape[1]:
            raise EOFError(f"feed exhausted at sample {self._next_sample}")
        block = self._data[:, self._next_sample:self._next_sample + n_samples]
        self._next_sample += n_samples
        return block


class MffFileFeed(SampleFeed):
    """Sequential reader over signal1.bin, `chunk_seconds` at a time.

    Reads only the two channels' byte ranges, using `MffSignalFile`'s block
    geometry.  Buffering a chunk ahead is I/O, not lookahead - the detector
    still only sees what `take` returns - and its time is reported separately.
    """

    def __init__(self, mff_dir, channel_names, start_second,
                 chunk_seconds=IO_CHUNK_SECONDS):
        self.signal_file = MffSignalFile(os.path.join(mff_dir, "signal1.bin"))
        file_rate = self.signal_file.sampling_rate
        if abs(file_rate - SAMPLING_RATE_HZ) > SAMPLING_RATE_TOLERANCE:
            raise ValueError(f"{mff_dir} is {file_rate} Hz, expected {SAMPLING_RATE_HZ}")
        gains = read_channel_gains(os.path.join(mff_dir, "info1.xml"))
        self.channel_indices = resolve_channel_indices(mff_dir, channel_names)
        self.to_volts = np.array(
            [MICROVOLTS_TO_VOLTS * (gains[index] if gains is not None else 1.0)
             for index in self.channel_indices])
        self.next_sample = int(round(start_second * SAMPLING_RATE_HZ))
        self.chunk_samples = int(chunk_seconds * SAMPLING_RATE_HZ)
        self.buffer = np.empty((len(channel_names), 0))
        self.io_sec = 0.0

    def take(self, n_samples):
        while self.buffer.shape[1] < n_samples:
            read_start = time.perf_counter()
            n_to_read = min(self.chunk_samples,
                            self.signal_file.n_samples - self.next_sample)
            if n_to_read <= 0:
                raise EOFError(f"end of {self.signal_file.path} at sample {self.next_sample}")
            raw_block = self.signal_file.read(
                self.channel_indices, self.next_sample,
                self.next_sample + n_to_read)
            self.io_sec += time.perf_counter() - read_start
            self.next_sample += n_to_read
            self.buffer = np.concatenate(
                [self.buffer,
                 raw_block.astype(np.float64) * self.to_volts[:, None]], axis=1)
        block, self.buffer = self.buffer[:, :n_samples], self.buffer[:, n_samples:]
        return block


# ----------------------------------------------------------------------
# A live feed would look like this.  Not implemented, not tested - a sketch
# of the ~30 lines you would fill in.
#
# import threading, collections
#
# class LiveFeed(SampleFeed):
#     """A ring buffer filled by an acquisition thread; `take` blocks."""
#
#     def __init__(self, n_chans, capacity_sec=60.0):
#         self._buf = collections.deque()          # of (n_chans, k) arrays
#         self._have = 0
#         self._lock = threading.Lock()
#         self._ready = threading.Condition(self._lock)
#         self._cap = int(capacity_sec * SAMPLING_RATE_HZ)
#
#     def push(self, block):                       # called by the amplifier
#         with self._ready:
#             self._buf.append(np.asarray(block, float))
#             self._have += block.shape[1]
#             if self._have > self._cap:
#                 raise RuntimeError("consumer fell behind the amplifier")
#             self._ready.notify()
#
#     def take(self, n):
#         with self._ready:
#             while self._have < n:
#                 self._ready.wait()               # <- the wall clock, live
#             out = np.concatenate(list(self._buf), axis=1)
#             self._buf.clear()
#             self._buf.append(out[:, n:])
#             self._have -= n
#             return out[:, :n]
#
# Missing before this touches a patient: a sequence number on every block, so
# a dropped packet is caught instead of silently concatenated (item 3 above),
# and a policy for when one turns up - reset the filters and re-seed, or stop.

---
## Reading the `.mff`

Straight out of `signal1.bin`, without MNE. `mne.io.read_raw_egi` walks every
block header before returning anything, then reads whole 257-channel blocks;
this reads two channels' byte ranges. On a Drive-backed recording that is ~30 MB
requested instead of ~3.8 GB.

Verified against MNE on a recording MNE could read: max |difference| < 1e-15 V.

In [ ]:
def read_channel_gains(info_xml_path):
    """The GCAL per-channel gain table out of infoN.xml, 1-based order."""
    with open(info_xml_path) as fh:
        text = fh.read()
    calibration_blocks = re.findall(r"<calibration>(.*?)</calibration>", text, re.S)
    for block in calibration_blocks:
        if "<type>GCAL</type>" not in block:
            continue
        channel_values = re.findall(r'<ch n="(\d+)">([^<]+)</ch>', block)
        gains = np.empty(len(channel_values))
        for channel_number, value in channel_values:
            gains[int(channel_number) - 1] = float(value)
        return gains
    return None


class MffSignalFile:
    """Block geometry of one signalN.bin, assuming a single uniform epoch."""

    def __init__(self, path):
        self.path = path
        self.size_bytes = os.path.getsize(path)
        with open(path, "rb") as fh:
            header = fh.read(MFF_HEADER_BYTES)
            version, self.header_size, self.block_size, self.n_channels = struct.unpack(
                "<4i", header)
            if version != 1:
                raise ValueError(f"first block is version {version}, expected 1")
            rest_of_header = fh.read(self.header_size - MFF_HEADER_BYTES)
        self.channel_offsets = np.frombuffer(
            rest_of_header, dtype="<i4", count=self.n_channels)
        sigfreq_words = np.frombuffer(
            rest_of_header, dtype="<i4", count=self.n_channels,
            offset=BYTES_PER_SAMPLE * self.n_channels)
        bit_depth = sigfreq_words[0] & MFF_DEPTH_MASK
        if bit_depth != MFF_BIT_DEPTH:
            raise ValueError(f"depth {bit_depth} != {MFF_BIT_DEPTH}")
        self.sampling_rate = float(sigfreq_words[0] >> MFF_RATE_SHIFT_BITS)
        self.samples_per_block = (self.block_size
                                  // BYTES_PER_SAMPLE // self.n_channels)

        # block 0 occupies header_size + block_size; every later block is a
        # 4-byte version==0 marker followed by block_size bytes of data
        self.block_stride_bytes = MFF_MARKER_BYTES + self.block_size
        self.first_block_end = self.header_size + self.block_size
        # How many blocks are certainly whole and uniform.  The last one is
        # usually short and may carry its own header, so drop it: it is the
        # last 4 s of a 6 h night.  check_markers() then reads the version word
        # of every block we touch, so a drifted stride shows up at once.
        self.n_blocks = (1 + (self.size_bytes - self.first_block_end)
                         // self.block_stride_bytes)
        self.tail_bytes = self.size_bytes - (self.first_block_end
                                       + (self.n_blocks - 1) * self.block_stride_bytes)
        if self.tail_bytes:
            # partial final block, geometry unknown - drop it
            self.n_blocks -= 1
        self.n_samples = self.n_blocks * self.samples_per_block

    def block_data_offset(self, block_index):
        """Byte offset of block k's DATA section (past its header/marker)."""
        if block_index == 0:
            return self.header_size
        return (self.first_block_end
                + (block_index - 1) * self.block_stride_bytes
                + MFF_MARKER_BYTES)

    def check_block_markers(self, block_indices):
        """Assert the given block indices really do start with version == 0."""
        with open(self.path, "rb") as fh:
            for block_index in block_indices:
                if block_index == 0:
                    continue
                fh.seek(self.first_block_end
                        + (block_index - 1) * self.block_stride_bytes)
                value = struct.unpack("<i", fh.read(MFF_MARKER_BYTES))[0]
                if value != 0:
                    raise ValueError(f"block {block_index} marker is {value}, expected 0")

    def read(self, channel_indices, start, stop, progress=None):
        """Samples [start, stop) of the given 0-based channel indices, raw units."""
        stop = min(stop, self.n_samples)
        samples_per_block = self.samples_per_block
        first_block = start // samples_per_block
        stop_block = (stop - 1) // samples_per_block + 1
        # Check the arithmetic against the file: an out-of-sync offset almost
        # never lands on a zero word.
        self.check_block_markers(sorted(
            {first_block, stop_block - 1,
             *range(first_block, stop_block, MFF_MARKER_CHECK_STRIDE)}))
        samples = np.empty((len(channel_indices), stop - start), dtype=np.float32)
        read_start = time.time()
        with open(self.path, "rb") as fh:
            for block_index in range(first_block, stop_block):
                block_offset = self.block_data_offset(block_index)
                block_first_sample = block_index * samples_per_block
                # the slice of this block the caller actually asked for
                from_sample = max(start, block_first_sample) - block_first_sample
                to_sample = (min(stop, block_first_sample + samples_per_block)
                             - block_first_sample)
                for row, channel in enumerate(channel_indices):
                    fh.seek(block_offset
                            + int(self.channel_offsets[channel])
                            + BYTES_PER_SAMPLE * from_sample)
                    raw_bytes = fh.read(BYTES_PER_SAMPLE * (to_sample - from_sample))
                    if len(raw_bytes) != BYTES_PER_SAMPLE * (to_sample - from_sample):
                        raise IOError(f"short read in block {block_index}, channel {channel}")
                    out_start = block_first_sample + from_sample - start
                    samples[row, out_start:out_start + to_sample - from_sample] = (
                        np.frombuffer(raw_bytes, dtype="<f4"))
                if progress and (block_index - first_block) % progress == 0:
                    done = block_index - first_block + 1
                    total = stop_block - first_block
                    elapsed = time.time() - read_start
                    print(f"  block {done}/{total}  {elapsed:6.0f}s elapsed, "
                          f"eta {elapsed / done * (total - done):6.0f}s", flush=True)
        return samples


def resolve_channel_indices(mff_dir, channel_names):
    """0-based index into signal1.bin for EGI names E1..E256 (VREF/Cz is 257)."""
    indices = []
    for channel_name in channel_names:
        if channel_name.upper() in ("VREF", "CZ"):
            indices.append(MFF_VREF_INDEX)
        else:
            match = re.fullmatch(r"[Ee](\d+)", channel_name)
            if not match:
                raise ValueError(f"cannot map channel name {channel_name!r}")
            indices.append(int(match.group(1)) - 1)
    return indices

---
## Features: 105 per channel, 210 in the row

The frozen classifier was trained on these exact definitions. **Do not "improve"
them.** `extract_epoch_features` is the notebook's `extract_epochs_features_mne`
with the DataFrame built once at the end: same calls, same values (max abs
difference 0.0), ~1.6x faster. The original is at the bottom of the notebook,
and checked against this one there.

Without the `create_filter` memo, `compute_energy_freq_bands` rebuilds six FIR
filters per window, ~30 % of feature time. It is a pure function, so caching
changes no value.

In [ ]:
# -------------------- mne.filter.create_filter cache --------------------
# `compute_energy_freq_bands` otherwise rebuilds 6 FIR filters per window,
# ~30 % of feature time.  They depend only on (sfreq, l_freq, h_freq, length),
# so cache them for every caller.
_create_filter_cache = {}
_original_create_filter = _mne_filter.create_filter


def _cached_create_filter(*args, **kwargs):
    def _positional_or_keyword(position, keyword):
        return args[position] if position < len(args) else kwargs.get(keyword)
    data = _positional_or_keyword(0, "data")
    sfreq = _positional_or_keyword(1, "sfreq")
    l_freq = _positional_or_keyword(2, "l_freq")
    h_freq = _positional_or_keyword(3, "h_freq")
    has_shape = hasattr(data, "shape") and getattr(data, "ndim", 0) > 0
    n_times = data.shape[-1] if has_shape else None
    other_kwargs = tuple(sorted(
        (k, repr(v)) for k, v in kwargs.items()
        if k not in ("data", "sfreq", "l_freq", "h_freq", "verbose")
    ))
    cache_key = (sfreq, l_freq, h_freq, n_times, other_kwargs)
    if cache_key not in _create_filter_cache:
        _create_filter_cache[cache_key] = _original_create_filter(*args, **kwargs)
    return _create_filter_cache[cache_key]


_mne_filter.create_filter = _cached_create_filter


_BAND_ORDER = ["theta", "alpha", "sigma", "beta", "gamma", "fast"]


def extract_epoch_features(epochs, subject, sampling_rate):
    """What extract_epochs_features_mne returns, computed faster.

    Same calls, same values, same column order.  It just collects arrays in a
    dict and builds the DataFrame once, instead of a column at a time.
    """
    univariate = mne_features.univariate   # every call below is one of these
    data = np.array(epochs)
    n_epochs = data.shape[0]

    ptp_amp = univariate.compute_ptp_amp(data)
    hjorth_mobility = univariate.compute_hjorth_mobility(data)
    hjorth_complexity = univariate.compute_hjorth_complexity(data)
    samp_entropy = univariate.compute_samp_entropy(data, emb=2, metric="chebyshev")
    spect_slope_arr = np.array(univariate.compute_spect_slope(
        sampling_rate, data, fmin=0.1, fmax=50, with_intercept=True, psd_method="welch", psd_params=None
    )).reshape(-1, 4)

    bands_dict = {
        "theta": (4, 8), "alpha": (8, 12), "sigma": (12, 16),
        "beta": (16, 30), "gamma": (30, 100), "fast": (100, 300),
    }
    energy_freq_bands_arr = np.array(univariate.compute_energy_freq_bands(
        sampling_rate, data, freq_bands=bands_dict
    )).reshape(-1, 6)
    app_entropy = univariate.compute_app_entropy(data, emb=2, metric="chebyshev")
    decorr_time = univariate.compute_decorr_time(sampling_rate, data)
    higuchi_fd = univariate.compute_higuchi_fd(data, kmax=10)
    hjorth_complexity_spect = univariate.compute_hjorth_complexity_spect(
        sampling_rate, data, normalize=False, psd_method="welch", psd_params=None
    )
    hjorth_mobility_spect = univariate.compute_hjorth_mobility_spect(
        sampling_rate, data, normalize=False, psd_method="welch", psd_params=None
    )
    hurst_exp = univariate.compute_hurst_exp(data)
    katz_fd = univariate.compute_katz_fd(data)
    kurtosis = univariate.compute_kurtosis(data)
    line_length = univariate.compute_line_length(data)
    mean = univariate.compute_mean(data)
    quantile = univariate.compute_quantile(data, q=0.75)
    rms = univariate.compute_rms(data)
    skewness = univariate.compute_skewness(data)
    spect_entropy = univariate.compute_spect_entropy(
        sampling_rate, data, psd_method="welch", psd_params=None)
    std = univariate.compute_std(data)
    svd_entropy = univariate.compute_svd_entropy(data, tau=2, emb=10)
    svd_fisher_info = univariate.compute_svd_fisher_info(data, tau=2, emb=10)
    variance = univariate.compute_variance(data)
    zero_crossings = univariate.compute_zero_crossings(
        data, threshold=2.220446049250313e-16)
    abspow = univariate.compute_pow_freq_bands(
        sampling_rate, data, {"total": (0.1, 500)}, False, psd_method="multitaper"
    )
    kaiser_arr = np.array(
        univariate.compute_teager_kaiser_energy(data)).reshape(-1, 12)
    pow_freq_bands_arr = np.array(univariate.compute_pow_freq_bands(
        data=epochs, sfreq=sampling_rate, freq_bands=bands_dict, normalize=True, ratios=None,
        ratios_triu=False, psd_method="multitaper", log=False, psd_params=None,
    )).reshape(-1, 6)
    wavelet_arr = np.array(univariate.compute_wavelet_coef_energy(
        data, wavelet_name="db4"
    )).reshape(-1, 5)

    feature_columns = {
        "subj": np.full(n_epochs, subject),
        "epoch_id": np.arange(n_epochs),
        "hjorth_mobility": np.asarray(hjorth_mobility).flatten(),
        "hjorth_complexity": np.asarray(hjorth_complexity).flatten(),
        "samp_entropy": np.asarray(samp_entropy).flatten(),
        "ptp_amp": np.asarray(ptp_amp).flatten(),
        "app_entropy": np.asarray(app_entropy).flatten(),
        "decorr_time": np.asarray(decorr_time).flatten(),
        "higuchi_fd": np.asarray(higuchi_fd).flatten(),
        "hjorth_complexity_spect": np.asarray(hjorth_complexity_spect).flatten(),
        "hjorth_mobility_spect": np.asarray(hjorth_mobility_spect).flatten(),
        "hurst_exp": np.asarray(hurst_exp).flatten(),
        "katz_fd": np.asarray(katz_fd).flatten(),
        "kurtosis": np.asarray(kurtosis).flatten(),
        "line_length": np.asarray(line_length).flatten(),
        "mean": np.asarray(mean).flatten(),
        "quantile": np.asarray(quantile).flatten(),
        "rms": np.asarray(rms).flatten(),
        "skewness": np.asarray(skewness).flatten(),
        "spect_entropy": np.asarray(spect_entropy).flatten(),
        "std": np.asarray(std).flatten(),
        "svd_entropy": np.asarray(svd_entropy).flatten(),
        "svd_fisher_info": np.asarray(svd_fisher_info).flatten(),
        "variance": np.asarray(variance).flatten(),
        "zero_crossings": np.asarray(zero_crossings).flatten(),
        "abspow_": np.asarray(abspow).flatten(),
        "spect_slope_intercept": spect_slope_arr[:, 0],
        "spect_slope_slope": spect_slope_arr[:, 1],
        "spect_slope_MSE": spect_slope_arr[:, 2],
        "spect_slope_R2": spect_slope_arr[:, 3],
    }

    band_position = {band: index for index, band in enumerate(_BAND_ORDER)}

    for index, band in enumerate(_BAND_ORDER):
        feature_columns[f"energy_freq_bands_{band}"] = energy_freq_bands_arr[:, index]
    for numerator_band in _BAND_ORDER:
        for denominator_band in _BAND_ORDER:
            if numerator_band != denominator_band:
                key = (f"energy_freq_bands_"
                       f"{numerator_band[0]}{denominator_band[0]}")
                feature_columns[key] = (
                    energy_freq_bands_arr[:, band_position[numerator_band]]
                    / energy_freq_bands_arr[:, band_position[denominator_band]]
                )

    for index in range(6):
        feature_columns[f"teager_kaiser_energy_{index}_mean"] = kaiser_arr[:, 2 * index]
        feature_columns[f"teager_kaiser_energy_{index}_std"] = (
            kaiser_arr[:, 2 * index + 1])

    for index, band in enumerate(_BAND_ORDER):
        feature_columns[f"pow_freq_bands_{band}"] = pow_freq_bands_arr[:, index]
    for numerator_band in _BAND_ORDER:
        for denominator_band in _BAND_ORDER:
            if numerator_band != denominator_band:
                key = f"pow_freq_bands_{numerator_band}/{denominator_band}"
                feature_columns[key] = (
                    pow_freq_bands_arr[:, band_position[numerator_band]]
                    / pow_freq_bands_arr[:, band_position[denominator_band]]
                )

    for index in range(5):
        feature_columns[f"wavelet_coef_energy_{index}"] = wavelet_arr[:, index]

    column_order = [
        "subj", "epoch_id", "app_entropy", "decorr_time",
        "energy_freq_bands_theta", "energy_freq_bands_alpha", "energy_freq_bands_sigma",
        "energy_freq_bands_beta", "energy_freq_bands_gamma", "energy_freq_bands_fast",
        "higuchi_fd", "hjorth_complexity", "hjorth_complexity_spect", "hjorth_mobility",
        "hjorth_mobility_spect", "hurst_exp", "katz_fd", "kurtosis", "line_length", "mean",
        "pow_freq_bands_theta", "pow_freq_bands_alpha", "pow_freq_bands_sigma",
        "pow_freq_bands_beta", "pow_freq_bands_gamma", "pow_freq_bands_fast",
        "pow_freq_bands_theta/alpha", "pow_freq_bands_theta/sigma", "pow_freq_bands_theta/beta",
        "pow_freq_bands_theta/gamma", "pow_freq_bands_theta/fast", "pow_freq_bands_alpha/theta",
        "pow_freq_bands_alpha/sigma", "pow_freq_bands_alpha/beta", "pow_freq_bands_alpha/gamma",
        "pow_freq_bands_alpha/fast", "pow_freq_bands_sigma/theta", "pow_freq_bands_sigma/alpha",
        "pow_freq_bands_sigma/beta", "pow_freq_bands_sigma/gamma", "pow_freq_bands_sigma/fast",
        "pow_freq_bands_beta/theta", "pow_freq_bands_beta/alpha", "pow_freq_bands_beta/sigma",
        "pow_freq_bands_beta/gamma", "pow_freq_bands_beta/fast", "pow_freq_bands_gamma/theta",
        "pow_freq_bands_gamma/alpha", "pow_freq_bands_gamma/sigma", "pow_freq_bands_gamma/beta",
        "pow_freq_bands_gamma/fast", "pow_freq_bands_fast/theta", "pow_freq_bands_fast/alpha",
        "pow_freq_bands_fast/sigma", "pow_freq_bands_fast/beta", "pow_freq_bands_fast/gamma",
        "ptp_amp", "quantile", "rms", "samp_entropy", "skewness", "spect_entropy",
        "spect_slope_intercept", "spect_slope_slope", "spect_slope_MSE", "spect_slope_R2",
        "std", "svd_entropy", "svd_fisher_info",
        "teager_kaiser_energy_0_mean", "teager_kaiser_energy_0_std",
        "teager_kaiser_energy_1_mean", "teager_kaiser_energy_1_std",
        "teager_kaiser_energy_2_mean", "teager_kaiser_energy_2_std",
        "teager_kaiser_energy_3_mean", "teager_kaiser_energy_3_std",
        "teager_kaiser_energy_4_mean", "teager_kaiser_energy_4_std",
        "teager_kaiser_energy_5_mean", "teager_kaiser_energy_5_std",
        "variance", "wavelet_coef_energy_0", "wavelet_coef_energy_1", "wavelet_coef_energy_2",
        "wavelet_coef_energy_3", "wavelet_coef_energy_4", "zero_crossings", "abspow_",
        "energy_freq_bands_ab", "energy_freq_bands_ag", "energy_freq_bands_as",
        "energy_freq_bands_af", "energy_freq_bands_at", "energy_freq_bands_bt",
        "energy_freq_bands_bs", "energy_freq_bands_bg", "energy_freq_bands_bf",
        "energy_freq_bands_st", "energy_freq_bands_sg", "energy_freq_bands_sf",
        "energy_freq_bands_gt", "energy_freq_bands_gf", "energy_freq_bands_ft",
    ]
    return pd.DataFrame({column: feature_columns[column] for column in column_order})

---
## Stage 1 and stage 3: the front end, one instance per channel

The 0.1 Hz corner is what forces the offline notebook into 33001 taps, so here
it is a causal IIR, which costs no lookahead. The 70 Hz low-pass is a short
symmetric FIR applied centred: phase-free, and it costs `(51-1)/2 = 25` samples
of future, the whole budget.

Then the notebook's 2 s box detrend, strictly **trailing**. The offline version
is centred and holds ~1 s of future; trailing costs nothing and is what every
number here was measured with.

In [ ]:
class CausalFilterChain:
    """0.1 Hz causal IIR -> 51-tap 70 Hz FIR -> trailing 2 s box.

    `bandpass` returns the causal z[m] = sum_j h[j] x[m-j].  The zero-phase
    output we want is y[n] = z[n + 25], so time n is ready once sample n + 25
    arrives; that 25-sample offset is the whole lookahead.  Filter state carries
    across calls, so streaming 250 at a time equals filtering the night at once.
    """

    def __init__(self, dc_offset=0.0, window_seconds=DETREND_WINDOW_SECONDS):
        self.fir_taps = sp_signal.firwin(FIR_TAP_COUNT, BANDPASS_HIGH_HZ,
                                         fs=SAMPLING_RATE_HZ)
        self.highpass_sos = sp_signal.butter(
            HIGHPASS_ORDER, BANDPASS_LOW_HZ, btype="highpass",
            output="sos", fs=SAMPLING_RATE_HZ)
        self.highpass_state = np.zeros((self.highpass_sos.shape[0], 2))
        self.fir_state = np.zeros(FIR_TAP_COUNT - 1)
        self.dc_offset = float(dc_offset)
        self.window_samples = int(SAMPLING_RATE_HZ * window_seconds)
        self.history = np.zeros(self.window_samples - 1)

    def bandpass(self, samples):
        centred = np.asarray(samples, float) - self.dc_offset
        filtered, self.highpass_state = sp_signal.sosfilt(
            self.highpass_sos, centred, zi=self.highpass_state)
        filtered, self.fir_state = sp_signal.lfilter(
            self.fir_taps, [1.0], filtered, zi=self.fir_state)
        return filtered

    def detrend(self, samples):
        """Subtract the mean of the trailing 2 s, zero-padded before the start.

        Divides by the full window even before it has filled, like
        `np.convolve(..., 'full')`, which is how every batch number was computed.
        """
        extended = np.concatenate([self.history, samples])
        cumulative = np.concatenate([[0.0], np.cumsum(extended)])
        trend = ((cumulative[self.window_samples:] - cumulative[:len(samples)])
                 / self.window_samples)
        self.history = extended[-(self.window_samples - 1):]
        return samples - trend

---
## Stage 6: the z-score, and where `chan_ptp` / `chan_skew` / `chan_kurt` come from

Statistics over the crop so far, **including** the window being decided, seeded
on the normalisation block. A running accumulator, so it costs nothing per epoch.

The offline notebook z-scores by the whole recording's mean and std, which is
the biggest thing a causal detector gives up. This expanding form is the best
self-contained option measured (`dev/RESULTS.md` §3). Only a per-channel
constant from an earlier night of the same patient beats it.

In [ ]:
class ExpandingNormaliser:
    """Statistics over the crop so far, including the window being decided.

    The offline `build_epochs(mode='expanding')` as a running accumulator: mean
    and std for the z-score, plus `chan_ptp` / `chan_skew` / `chan_kurt`.  Best
    of the self-contained options measured (`dev/RESULTS.md` section 3).
    """

    def __init__(self):
        self._n_samples = 0
        self._moment_sums = np.zeros(N_RAW_MOMENTS)
        self._minimum, self._maximum = np.inf, -np.inf

    def observe(self, samples):
        samples = np.asarray(samples, float)
        self._n_samples += len(samples)
        self._moment_sums += [samples.sum(), (samples ** 2).sum(),
                              (samples ** 3).sum(), (samples ** 4).sum()]
        self._minimum = min(self._minimum, samples.min())
        self._maximum = max(self._maximum, samples.max())

    def current_stats(self):
        n_samples = max(self._n_samples, 1)
        (raw_moment_1, raw_moment_2,
         raw_moment_3, raw_moment_4) = self._moment_sums / n_samples
        mean = raw_moment_1
        variance = max(raw_moment_2 - mean ** 2, VARIANCE_FLOOR)
        std_dev = np.sqrt(variance)
        third_central = raw_moment_3 - 3 * mean * raw_moment_2 + 2 * mean ** 3
        fourth_central = (raw_moment_4 - 4 * mean * raw_moment_3
                          + 6 * mean ** 2 * raw_moment_2 - 3 * mean ** 4)
        return mean, std_dev, {
            "chan_ptp": (self._maximum - self._minimum) / std_dev,
            "chan_skew": third_central / variance ** 1.5,
            "chan_kurt": fourth_central / variance ** 2 - 3.0}

---
## Stage 7: threshold, refractory, bins

Fire on the first epoch at or above threshold, then skip 2 s. The refractory
restarts at every 20 min bin boundary.

`drop_short_final` throws away the last piece of the night if it is too short.
The night is cut into 20 min bins, and the last piece rarely fits. A 43.5 min
crop is two full bins plus a leftover:

```
|---- bin 1: 20 min ----|---- bin 2: 20 min ----|-- 3.5 min --|
```

The leftover is under half a bin, so with the flag on, scoring stops at minute
40 and nothing in the tail can fire.

That is the one subtlety here, and it caused a long-standing discrepancy. The
batch scorer drops such a leftover, so on the crop above it reports 209
detections where a streaming detector emits 291. The signal chain is not
involved. The two normalisers agree to 1e-14 and the detrended samples are
bit-equal.

* `True` - the batch scorer. Use it to match published numbers on a fixed crop.
* `False` - correct for a live device, which should not go silent over the last
  few minutes of a night. **The default here.**


In [ ]:
class ThresholdDetector:
    """Stage 7 as a per-epoch state machine.

    `step` returns the detection time in seconds, or None.  Fed
    epochs in order it emits exactly what the batch scorer does for the same
    probabilities, short final bin and all, when `n_epochs` is known and
    `drop_short_final` is on.
    """

    def __init__(self, detection_threshold=DETECTION_THRESHOLD,
                 refractory_seconds=REFRACTORY_SECONDS,
                 n_epochs=None, drop_short_final=True):
        self.detection_threshold = float(detection_threshold)
        self.refractory_epochs = int(refractory_seconds * EPOCHS_PER_SECOND)
        self.epochs_to_skip = 0
        self.detection_times = []
        self.n_epochs = n_epochs
        # drop_short_final throws away the last bin of the night when it's
        # too short.  This marks where that bin starts; nothing from there on
        # fires.  None = every bin counts.
        self.first_dropped_epoch = None
        if drop_short_final and n_epochs:
            last_full_bin_end = (n_epochs // EPOCHS_PER_BIN) * EPOCHS_PER_BIN
            if n_epochs - last_full_bin_end < EPOCHS_PER_BIN // 2:
                self.first_dropped_epoch = last_full_bin_end

    def step(self, epoch_index, probability):
        if (self.first_dropped_epoch is not None
                and epoch_index >= self.first_dropped_epoch):
            return None
        if epoch_index % EPOCHS_PER_BIN == 0:
            self.epochs_to_skip = 0
        if self.epochs_to_skip > 0:
            self.epochs_to_skip -= 1
            return None
        if probability < self.detection_threshold:
            return None
        self.epochs_to_skip = self.refractory_epochs - 1
        detection_time = round(epoch_index * EPOCH_SECONDS, TIME_DECIMALS)
        self.detection_times.append(detection_time)
        return detection_time

---
## The two models

`models/xgb_frozen.pkl` is model 1, the classifier from the original offline
notebook. **Frozen**: never refit, never re-saved. It was pickled by an older
XGBoost, so loading it warns; that is suppressed above.

`models/correction.json` is model 2, fitted by `dev/train.ipynb` on every NREM
period of the three donor recordings.

Both go through `Booster.inplace_predict` on a **preallocated** row. The sklearn
`predict_proba` wrapper spends ~49 ms per call validating feature names, four
times the whole compute budget; a device using it would be measuring pandas.

(`epochs_to_features` in the original package forks a pool, and a booster loaded
before a fork can deadlock the children. Nothing here forks.)

In [ ]:
def build_design_matrix(per_channel_frames):
    """The batch layout: two per-channel frames -> one 210-column row.

    Used only by the warm-up check; the live loop uses `FeatureRowBuilder`.
    """
    channel_1 = per_channel_frames[ANALYSIS_CHANNELS[0]]
    channel_2 = per_channel_frames[ANALYSIS_CHANNELS[1]]
    row = pd.concat([channel_1, channel_2], axis=1, ignore_index=True)
    row.columns = ([f"eog1_{column}" for column in channel_1.columns]
                   + [f"eog2_{column}" for column in channel_2.columns])
    return row[FROZEN_MODEL.get_booster().feature_names]


WHOLE_CHANNEL_COLUMNS = ("chan_ptp", "chan_skew", "chan_kurt")


class FeatureRowBuilder:
    """`build_design_matrix` for a single epoch, as a precomputed gather.

    `XGBClassifier.predict_proba` spends ~49 ms a call validating feature names,
    forty times the prediction itself and the biggest term in the latency.  The
    column layout never changes, so it is resolved once here and each epoch is
    a numpy gather plus `Booster.inplace_predict`.  `warm_up_and_verify` checks the
    two paths agree.
    """

    def __init__(self):
        feature_names = FROZEN_MODEL.get_booster().feature_names
        # every model feature is "eog1_<name>" or "eog2_<name>"
        base_names = [name[CHANNEL_PREFIX_LEN:] for name in feature_names]
        channel_of = [0 if name.startswith("eog1_") else 1
                      for name in feature_names]
        self.epoch_feature_columns = list(dict.fromkeys(
            base_name for base_name in base_names
            if base_name not in WHOLE_CHANNEL_COLUMNS))
        column_position = {column: position for position, column
                           in enumerate(self.epoch_feature_columns)}
        self.output_positions = np.array(
            [index for index, base_name in enumerate(base_names)
             if base_name not in WHOLE_CHANNEL_COLUMNS])
        self.source_rows = np.array(
            [channel_of[index] for index in self.output_positions])
        self.source_columns = np.array(
            [column_position[base_names[index]]
             for index in self.output_positions])
        self.whole_channel_plan = [(index, channel_of[index], base_names[index])
                                   for index, base_name in enumerate(base_names)
                                   if base_name in WHOLE_CHANNEL_COLUMNS]
        self.row = np.empty((1, len(feature_names)))

    def __call__(self, epoch_features, whole_channel_stats):
        """`epoch_features`: the 2-row frame `extract_epoch_features` returns,
        one row per channel.  `whole_channel_stats`: one
        {chan_ptp, chan_skew, chan_kurt} dict per channel."""
        values = epoch_features[self.epoch_feature_columns].to_numpy(dtype=np.float64)
        self.row[0, self.output_positions] = values[self.source_rows,
                                                    self.source_columns]
        for index, channel, column in self.whole_channel_plan:
            self.row[0, index] = whole_channel_stats[channel][column]
        return self.row

In [ ]:
"""Load both boosters.  `FROZEN_MODEL` must exist before `FeatureRowBuilder`
is built, because the column plan is read off its feature names."""
FROZEN_MODEL = joblib.load(os.path.join(MODELS_DIR, "xgb_frozen.pkl"))
FROZEN_BOOSTER = FROZEN_MODEL.get_booster()
assert len(FROZEN_BOOSTER.feature_names) == N_MODEL_FEATURES

with open(os.path.join(MODELS_DIR, "correction_meta.json")) as fh:
    CORRECTION_META = json.load(fh)
CORRECTION_BOOSTER = xgb.Booster()
CORRECTION_BOOSTER.load_model(os.path.join(MODELS_DIR, "correction.json"))
CORRECTION_ALPHA = float(CORRECTION_META["alpha"])

# the correction was picked at this detection_threshold, on this chain, with this
# normalisation block; if any has moved, it no longer applies
assert CORRECTION_META["threshold"] == DETECTION_THRESHOLD
assert CORRECTION_META["sec_limit"] == REFRACTORY_SECONDS
assert CORRECTION_META["bin_length_min"] == BIN_LENGTH_MINUTES
assert CORRECTION_META["calib_min"] == NORMALISATION_MINUTES
assert CORRECTION_META["lookahead_ms"] == LOOKAHEAD_MS
assert CORRECTION_META["chain"] == "b25_o3" and CORRECTION_META["norm"] == "expand20"
assert CORRECTION_META["n_features"] == N_MODEL_FEATURES + 1      # + the frozen logit

print(f"model 1  frozen classifier, {len(FROZEN_BOOSTER.feature_names)} features")
print(f"model 2  correction, depth {CORRECTION_META['depth']}, alpha {CORRECTION_ALPHA:.2f}, "
      f"fitted {CORRECTION_META['created']}")
print(f"         on {CORRECTION_META['n_windows']} NREM windows / "
      f"{CORRECTION_META['train_min']:.0f} min / {CORRECTION_META['n_rows']} rows")
print(f"         honest estimate for a new patient: "
      f"p {CORRECTION_META['loso_estimate']['mean_precision']:.3f} / "
      f"r {CORRECTION_META['loso_estimate']['mean_recall']:.3f} / "
      f"F1 {CORRECTION_META['loso_estimate']['mean_f1']:.3f}   (leave-one-recording-out)")
print("         constants.py agrees with the metadata on threshold, refractory,\n"
      "         bins, normalisation, lookahead and feature count")

---
## The live loop

One object: consume the warm-up and the normalisation block, then answer once per
250 ms window. Every window is timed into `filter_ms` / `feature_ms` /
`model1_ms` / `model2_ms` / `compute_ms`, always. It costs a `perf_counter`
call.

"Compute" is charged the whole window: filtering, detrending and normalising all
250 samples is timed here even though a live device does it as they arrive. It
is an upper bound.

In [ ]:
class StreamingDetector:
    """Stream a recording and answer 25 ms + compute after each window.

        detector = StreamingDetector(feed, warmup_samples, n_crop_samples,
                                     correction=CORRECTION_BOOSTER,
                                     alpha=CORRECTION_ALPHA)
        detector.fit_normalisation()   # warm-up + the normalisation block
        result = detector.run()        # result["fired"] = [(sec, proba), ...]

    `correction=None` runs the frozen booster alone - the published baseline.
    """

    def __init__(self, feed, warmup_samples, n_crop_samples,
                 normalisation_seconds=NORMALISATION_SECONDS,
                 detection_threshold=DETECTION_THRESHOLD, correction=None, alpha=0.0,
                 drop_short_final=False):
        self.feed = feed
        # samples of filter warm-up read before the crop
        self.warmup_samples = int(warmup_samples)
        self.n_crop_samples = int(n_crop_samples)
        self.normalisation_seconds = float(normalisation_seconds)
        self.normalisation_samples = int(self.normalisation_seconds * SAMPLING_RATE_HZ)
        self.threshold = float(detection_threshold)
        self.correction = correction
        self.alpha = float(alpha)
        self.drop_short_final = bool(drop_short_final)
        self.filter_chains = [CausalFilterChain() for _ in ANALYSIS_CHANNELS]
        self.normalisers = [ExpandingNormaliser() for _ in ANALYSIS_CHANNELS]
        # stage 6 drops the last four epochs' worth of samples, so a 51 min
        # crop is 12237 epochs, not 12240
        self.n_epochs = len(range(0, self.n_crop_samples
                                  - DROPPED_TAIL_EPOCHS * EPOCH_SAMPLES, EPOCH_SAMPLES))
        self.filter_ms, self.feature_ms = [], []
        self.model1_ms, self.model2_ms, self.compute_ms = [], [], []

    # ---------------- normalisation ----------------
    def fit_normalisation(self, verbose=True):
        """Consume the warm-up and the normalisation block.

        Two products, both from past samples only: the DC each band-pass
        removes (the mean so far - the IIR owns the 0.1 Hz corner and needs a
        centred input), and the seed of the expanding normaliser.

        Nothing fires during the block, so it can be processed in one go.
        """
        start_time = time.time()
        block = self.feed.take(self.warmup_samples
                               + self.normalisation_samples + LOOKAHEAD_SAMPLES)
        detrended = np.empty((len(ANALYSIS_CHANNELS), self.normalisation_samples))
        for channel in range(len(ANALYSIS_CHANNELS)):
            # excludes the lookahead samples of the first scored window, so
            # the constant is exactly the batch one
            past = block[channel,
                         :self.warmup_samples + self.normalisation_samples]
            self.filter_chains[channel].dc_offset = float(past.mean())
            bandpassed = self.filter_chains[channel].bandpass(block[channel])
            # crop-aligned: drop the warm-up and the lookahead
            detrended[channel] = self.filter_chains[channel].detrend(
                bandpassed[self.warmup_samples + LOOKAHEAD_SAMPLES:])
            self.normalisers[channel].observe(detrended[channel])
        if verbose:
            print(f"  normalised on "
                  f"{self.normalisation_seconds / SECONDS_PER_MINUTE:g} min "
                  f"({time.time() - start_time:.0f}s): DC "
                  + "/".join(f"{self.filter_chains[channel].dc_offset:+.2e}"
                             for channel in range(len(ANALYSIS_CHANNELS)))
                  + " V, z-score sd "
                  + "/".join(f"{self.normalisers[channel].current_stats()[1]:.2e}"
                             for channel in range(len(ANALYSIS_CHANNELS))), flush=True)

    # ---------------- the live loop ----------------
    def run(self, verbose=True, on_epoch=None):
        normalisation_epochs = self.normalisation_samples // EPOCH_SAMPLES
        threshold_detector = ThresholdDetector(
            self.threshold, n_epochs=self.n_epochs,
            drop_short_final=self.drop_short_final)
        frozen_booster = FROZEN_BOOSTER      # local, looked up 12k times
        row_builder = FeatureRowBuilder()
        probabilities = np.full(self.n_epochs, np.nan)
        fired = []
        # one throwaway prediction each, outside the timed loop, so the first
        # timed epoch doesn't measure xgboost's first-call setup
        warm_up_and_verify(frozen_booster, row_builder)
        correction_row = np.empty((1, row_builder.row.shape[1] + 1))
        if self.correction is not None:
            correction_row[0, :-1] = row_builder.row[0]
            correction_row[0, -1] = 0.0
            self.correction.inplace_predict(correction_row)

        start_time = time.time()
        for epoch_index in range(normalisation_epochs, self.n_epochs):
            # ---- the window closes here; the next 25 samples are all the
            # ---- lookahead the chain gets
            window = self.feed.take(EPOCH_SAMPLES)
            # every *_done_at below is a perf_counter reading in seconds;
            # only the differences are kept, in ms
            window_opened_at = time.perf_counter()

            normalised_segments, whole_channel_stats = [], []
            for channel in range(len(ANALYSIS_CHANNELS)):
                bandpassed = self.filter_chains[channel].bandpass(window[channel])
                segment = self.filter_chains[channel].detrend(bandpassed)
                self.normalisers[channel].observe(segment)
                mean, std_dev, channel_stats = self.normalisers[channel].current_stats()
                normalised_segments.append((segment - mean) / std_dev)
                whole_channel_stats.append(channel_stats)
            filter_done_at = time.perf_counter()

            # both channels in one extractor call: per-call overhead
            # dominates at a single epoch, so two calls cost nearly twice
            epoch_features = extract_epoch_features(
                np.stack(normalised_segments), "live", SAMPLING_RATE_HZ)
            features_done_at = time.perf_counter()

            feature_row = row_builder(epoch_features, whole_channel_stats)
            frozen_probability = frozen_booster.inplace_predict(feature_row)[0]
            model1_done_at = time.perf_counter()

            if self.correction is None:
                probability = frozen_probability
            else:
                frozen_logit = np.log(
                    max(frozen_probability, PROBABILITY_CLIP_EPS)
                    / max(1.0 - frozen_probability, PROBABILITY_CLIP_EPS))
                correction_row[0, :-1] = feature_row[0]
                correction_row[0, -1] = frozen_logit
                probability = 1.0 / (1.0 + np.exp(
                    -(frozen_logit + self.alpha
                      * self.correction.inplace_predict(correction_row)[0])))
            probabilities[epoch_index] = probability
            detection_time = threshold_detector.step(epoch_index, probability)
            model2_done_at = time.perf_counter()

            self.filter_ms.append(
                (filter_done_at - window_opened_at) * MILLISECONDS_PER_SECOND)
            self.feature_ms.append(
                (features_done_at - filter_done_at) * MILLISECONDS_PER_SECOND)
            self.model1_ms.append(
                (model1_done_at - features_done_at) * MILLISECONDS_PER_SECOND)
            self.model2_ms.append(
                (model2_done_at - model1_done_at) * MILLISECONDS_PER_SECOND)
            self.compute_ms.append(
                (model2_done_at - window_opened_at) * MILLISECONDS_PER_SECOND)
            if detection_time is not None:
                fired.append((detection_time, float(probability)))
            if on_epoch is not None:
                on_epoch(epoch_index, probability, detection_time)
            if verbose and ((epoch_index - normalisation_epochs) % PROGRESS_EVERY_N_EPOCHS
                            == PROGRESS_EVERY_N_EPOCHS - 1):
                print(f"    epoch {epoch_index - normalisation_epochs + 1}/"
                      f"{self.n_epochs - normalisation_epochs}  {len(fired)} spikes  "
                      f"{time.time() - start_time:.0f}s", flush=True)

        return {"proba": probabilities, "fired": fired,
                "times": threshold_detector.detection_times,
                "n_epochs": self.n_epochs, "n_norm_epochs": normalisation_epochs,
                "seconds": time.time() - start_time}


def warm_up_and_verify(frozen_booster, row_builder):
    """First prediction out of the timed loop, and a check that the fast
    gather matches `build_design_matrix`."""
    noise_epochs = np.random.default_rng(0).normal(
        size=(len(ANALYSIS_CHANNELS), EPOCH_SAMPLES))
    epoch_features = extract_epoch_features(noise_epochs, "warmup", SAMPLING_RATE_HZ)
    whole_channel_stats = [{"chan_ptp": 7.0 + channel,
                            "chan_skew": -0.3 * channel,
                            "chan_kurt": 4.0 - channel}
                           for channel in range(len(ANALYSIS_CHANNELS))]
    frames = {}
    for channel, channel_name in enumerate(ANALYSIS_CHANNELS):
        frame = epoch_features.iloc[[channel]].reset_index(drop=True)
        frame["chan_name"] = channel_name
        for column, value in whole_channel_stats[channel].items():
            frame[column] = value
        frames[channel_name] = frame
    proba_via_dataframe = FROZEN_MODEL.predict_proba(
        build_design_matrix(frames))[:, 1][0]
    proba_via_gather = frozen_booster.inplace_predict(
        row_builder(epoch_features, whole_channel_stats))[0]
    assert abs(proba_via_dataframe - proba_via_gather) < WARMUP_MATCH_TOLERANCE, (
        proba_via_dataframe, proba_via_gather)


def make_mff_feed(mff_dir, start_minute, chunk_seconds=IO_CHUNK_SECONDS):
    """A `SampleFeed` reading E227 / E254 straight out of `<mff>/signal1.bin`."""
    return MffFileFeed(mff_dir, ANALYSIS_CHANNELS,
                       start_minute * SECONDS_PER_MINUTE, chunk_seconds)


def make_array_feed(data, start_sample=0):
    """A `SampleFeed` over an in-memory `(len(ANALYSIS_CHANNELS), n_samples)` array."""
    return InMemoryFeed(data, start_sample)

---
# Run

Reads the parameters from the cell at the top; everything it needs is defined
above. The banner it prints is what the run actually used.

In [ ]:
"""Run the detector.  ~1 s of wall clock per 35-40 s of recording on 8 cores."""
if not os.path.isdir(MFF_DIR):
    raise SystemExit(f"MFF is not a directory: {MFF_DIR}\n"
                     f"Point it at the .mff folder holding signal1.bin.")
if END_MINUTE - START_MINUTE <= NORMALISATION_MINUTES:
    raise SystemExit(f"the analysed span is {END_MINUTE - START_MINUTE:g} min and the "
                     f"normalisation block alone is {NORMALISATION_MINUTES:g} min - nothing "
                     f"would be scored")

print(f"recording     {os.path.basename(os.path.realpath(MFF_DIR))}")
print(f"span          {START_MINUTE:g} - {END_MINUTE:g} min ({END_MINUTE - START_MINUTE:g} min), "
      f"{FILTER_WARMUP_MINUTES:g} min warm-up")
print(f"chain         b25_o3 x expand20, {LOOKAHEAD_MS:.0f} ms lookahead, "
      f"channels {'/'.join(ANALYSIS_CHANNELS)}")
print("model         " + (f"frozen booster + correction (depth "
                          f"{CORRECTION_META['depth']}, alpha {CORRECTION_ALPHA:.2f})"
                          if USE_CORRECTION else
                          "frozen booster only (USE_CORRECTION = False)"))
print(f"threshold     {DETECTION_THRESHOLD:.2f}, {REFRACTORY_SECONDS} s refractory, "
      f"{BIN_LENGTH_MINUTES} min bins")
print(f"normalisation {NORMALISATION_MINUTES:g} min - no detections before minute "
      f"{START_MINUTE + NORMALISATION_MINUTES:g}\n", flush=True)

detector = StreamingDetector(
    make_mff_feed(MFF_DIR, START_MINUTE - FILTER_WARMUP_MINUTES),
    warmup_samples=int(round(FILTER_WARMUP_MINUTES * SECONDS_PER_MINUTE * SAMPLING_RATE_HZ)),
    # + 1 because the offline notebook's stage 2 crops data[a:b + 1]
    n_crop_samples=int(round((END_MINUTE - START_MINUTE)
                             * SECONDS_PER_MINUTE * SAMPLING_RATE_HZ)) + 1,
    normalisation_seconds=NORMALISATION_SECONDS,
    detection_threshold=DETECTION_THRESHOLD,
    correction=CORRECTION_BOOSTER if USE_CORRECTION else None,
    alpha=CORRECTION_ALPHA if USE_CORRECTION else 0.0,
    drop_short_final=DROP_SHORT_FINAL,
)
_run_start = time.time()
detector.fit_normalisation()
run_result = detector.run()
print(f"\ndone in {time.time() - _run_start:.0f}s")

In [ ]:
"""Results, the latency budget, and the CSV."""
fired_detections = run_result["fired"]
scored_minutes = (END_MINUTE - START_MINUTE) - NORMALISATION_MINUTES
compute_ms = np.array(detector.compute_ms)
end_to_end_ms = compute_ms + LOOKAHEAD_MS

print(f"{len(fired_detections)} detections over {scored_minutes:g} scored min "
      f"({len(fired_detections) / scored_minutes:.2f} per min)\n")
print(f"{'':14s}{'median':>9s}{'mean':>9s}{'p95':>9s}{'max':>9s}")
for label, durations_ms in (("filter+norm", np.array(detector.filter_ms)),
                            ("features", np.array(detector.feature_ms)),
                            ("model 1", np.array(detector.model1_ms)),
                            ("model 2", np.array(detector.model2_ms)),
                            ("compute", compute_ms),
                            ("END-TO-END", end_to_end_ms)):
    print(f"{label:14s}{np.median(durations_ms):8.1f} {durations_ms.mean():8.1f} "
          f"{np.percentile(durations_ms, LATENCY_PERCENTILE):8.1f} {durations_ms.max():8.1f}  ms")
_budget_ms = EPOCH_SECONDS * MILLISECONDS_PER_SECOND
_over_budget = int((end_to_end_ms > _budget_ms).sum())
print(f"\nEND-TO-END = {LOOKAHEAD_MS:.0f} ms of signal lookahead + compute.  "
      f"The budget is one epoch, {_budget_ms:.0f} ms;\n"
      f"{_over_budget} of {len(end_to_end_ms)} windows went over it "
      f"({100 * _over_budget / len(end_to_end_ms):.2f} %).")
print(f"{run_result['seconds']:.0f}s of compute for {scored_minutes:g} scored min "
      f"({scored_minutes * SECONDS_PER_MINUTE / max(run_result['seconds'], DIVISION_GUARD):.0f}x real time; "
      f"the normalisation block is not counted).")

# detection times are seconds from START_MINUTE; add it back so they index the recording
_offset_seconds = START_MINUTE * SECONDS_PER_MINUTE
detections = pd.DataFrame({
    "time_sec": [round(detection_time + _offset_seconds, TIME_DECIMALS)
                 for detection_time, _ in fired_detections],
    "time_min": [round((detection_time + _offset_seconds) / SECONDS_PER_MINUTE,
                       MINUTE_DECIMALS)
                 for detection_time, _ in fired_detections],
    "probability": [round(probability, PROBABILITY_DECIMALS)
                    for _, probability in fired_detections]})
if OUTPUT_CSV:
    detections.to_csv(OUTPUT_CSV, index=False)
    print(f"\n-> {OUTPUT_CSV}")
detections.head(PREVIEW_ROWS)

---
# Verification

Three checks. The first and third run anywhere; the second needs a specific
recording and skips itself cleanly when it is absent.

In [ ]:
"""Expected values, recorded when this notebook was built and verified.

The two time lists are what the research package emitted on the AO0 recording, minutes
4-47.5.  The notebook is correct only if it reproduces them exactly.
"""
SYNTHETIC_EXPECTED = {
    "n_epochs": 476,
    "proba_sum": 196.5159,
    "proba_max": 0.917924,
    "n_detections": 19
}

ACCEPTANCE_FROZEN_TIMES = [1682.00, 1702.75, 1710.25, 1721.25, 1726.50,
    1739.50, 1753.50, 1775.25, 1787.75, 1794.50, 1798.00, 1803.00, 1805.25,
    1807.75, 1816.00, 1818.50, 1820.75, 1838.00, 1842.75, 1846.25, 1851.25,
    1853.25, 1882.75, 1886.50, 1895.50, 1898.25, 1906.25, 1915.50, 1923.25,
    1925.50, 1958.25, 1964.25, 1971.00, 1980.50, 1983.25, 1987.50, 1998.50,
    2001.00, 2005.75, 2014.00, 2016.50, 2025.25, 2031.00, 2040.00, 2042.00,
    2044.75, 2048.25, 2051.50, 2056.75, 2063.50, 2068.75, 2072.75, 2078.00,
    2082.00, 2085.25, 2088.00, 2092.50, 2094.50, 2096.50, 2099.75, 2103.75,
    2112.25, 2114.75, 2133.25, 2141.50, 2144.00, 2148.00, 2161.25, 2163.25,
    2167.50, 2169.50, 2171.50, 2176.00, 2180.00, 2182.25, 2184.25, 2187.00,
    2190.75, 2193.50, 2199.25, 2202.50, 2205.25, 2209.00, 2213.25, 2216.00,
    2218.75, 2222.75, 2225.25, 2230.00, 2232.00, 2234.00, 2237.00, 2239.25,
    2241.50, 2245.25, 2255.50, 2258.50, 2266.75, 2270.25, 2276.50, 2278.50,
    2280.50, 2282.50, 2285.50, 2288.00, 2292.00, 2294.00, 2297.50, 2310.50,
    2313.50, 2317.00, 2320.25, 2322.25, 2325.25, 2327.50, 2330.50, 2332.75,
    2335.00, 2337.00, 2339.75, 2342.25, 2346.75, 2349.75, 2351.75, 2357.00,
    2360.50, 2363.50, 2367.00, 2371.00, 2375.00, 2377.25, 2380.50, 2383.50,
    2388.25, 2391.00, 2393.25, 2395.50, 2400.50, 2404.00, 2416.75, 2420.00,
    2422.50, 2425.50, 2429.00, 2432.00, 2434.00, 2436.75, 2438.75, 2440.75,
    2444.75, 2446.75, 2449.25, 2453.00, 2455.25, 2457.50, 2461.25, 2464.25,
    2468.00, 2471.50, 2474.75, 2481.00, 2483.00, 2485.25, 2487.25, 2490.00,
    2492.25, 2494.25, 2502.50, 2504.75, 2506.75, 2509.00, 2511.00, 2515.25,
    2517.50, 2519.50, 2521.50, 2523.50, 2525.50, 2527.50, 2534.25, 2546.00,
    2548.00, 2553.25, 2555.25, 2558.00, 2563.25, 2575.00, 2577.75, 2580.25,
    2582.50, 2585.50, 2587.50, 2591.75, 2594.50, 2598.50, 2601.25, 2603.25,
    2606.00, 2608.75, 2614.50, 2618.75, 2621.50, 2624.75, 2627.50, 2629.75,
    2631.75, 2633.75, 2636.25, 2638.25, 2640.00, 2642.50, 2646.00, 2650.25,
    2652.25, 2655.50, 2659.00, 2661.25, 2664.50, 2666.50, 2669.75, 2672.25,
    2674.50, 2677.00, 2679.00, 2681.50, 2683.50, 2685.50, 2687.50, 2690.00,
    2692.50, 2695.00, 2697.75, 2701.00, 2703.25, 2705.25, 2707.75, 2709.75,
    2713.00, 2715.00, 2717.75, 2720.50, 2724.50, 2728.00, 2730.00, 2732.25,
    2734.25, 2736.25, 2738.25, 2740.25, 2743.25, 2745.25, 2750.25, 2752.75,
    2755.50, 2758.00, 2760.00, 2762.50, 2766.00, 2769.25, 2771.50, 2774.75,
    2776.75, 2778.75, 2782.00, 2784.00, 2786.75, 2789.75, 2793.25, 2795.75,
    2797.75, 2800.00, 2802.00, 2804.00, 2806.00, 2810.25, 2812.50, 2814.50,
    2818.25, 2820.25, 2822.50, 2825.00, 2827.25, 2829.25, 2831.25, 2834.00,
    2836.00, 2838.00, 2840.00, 2842.50, 2844.50, 2846.75]

ACCEPTANCE_CORRECTED_TIMES = [1682.00, 1684.25, 1710.25, 1713.00, 1721.25,
    1726.50, 1732.50, 1739.50, 1753.50, 1775.25, 1787.75, 1794.50, 1803.00,
    1805.25, 1807.75, 1811.25, 1818.25, 1821.25, 1832.00, 1838.00, 1843.00,
    1846.25, 1851.25, 1853.25, 1882.75, 1886.50, 1894.75, 1896.75, 1898.75,
    1903.50, 1906.25, 1915.50, 1923.25, 1925.25, 1964.25, 1971.00, 1980.75,
    1983.25, 1987.25, 1999.25, 2001.50, 2005.75, 2014.00, 2016.50, 2023.75,
    2031.00, 2040.00, 2042.00, 2044.75, 2048.25, 2051.50, 2054.25, 2056.75,
    2063.50, 2067.00, 2069.50, 2073.00, 2076.75, 2079.50, 2082.25, 2085.25,
    2088.00, 2093.00, 2096.50, 2099.75, 2103.75, 2114.25, 2141.50, 2144.00,
    2147.50, 2161.25, 2163.25, 2165.25, 2168.25, 2171.00, 2175.75, 2180.00,
    2182.25, 2185.00, 2187.00, 2192.00, 2199.25, 2205.25, 2209.00, 2213.25,
    2216.50, 2218.75, 2225.25, 2230.25, 2232.75, 2235.50, 2237.50, 2239.50,
    2250.25, 2255.50, 2258.75, 2276.50, 2278.50, 2280.50, 2282.50, 2285.50,
    2288.25, 2292.00, 2294.00, 2311.00, 2314.50, 2317.00, 2322.25, 2325.25,
    2327.75, 2330.50, 2332.75, 2335.25, 2337.25, 2342.75, 2347.00, 2349.75,
    2352.00, 2357.00, 2360.50, 2364.25, 2367.00, 2371.00, 2375.00, 2377.25,
    2380.50, 2383.50, 2388.25, 2391.25, 2393.25, 2395.50, 2404.00, 2417.00,
    2420.00, 2422.50, 2425.50, 2429.00, 2432.00, 2434.00, 2436.75, 2439.25,
    2441.75, 2444.75, 2446.75, 2449.25, 2454.00, 2456.00, 2458.25, 2461.50,
    2464.25, 2468.00, 2471.75, 2474.75, 2481.00, 2483.00, 2485.00, 2487.00,
    2490.00, 2492.25, 2494.25, 2502.75, 2504.75, 2506.75, 2509.00, 2511.00,
    2515.25, 2517.50, 2521.00, 2523.50, 2525.50, 2527.50, 2534.25, 2548.00,
    2553.25, 2555.25, 2563.25, 2575.00, 2578.00, 2580.25, 2582.50, 2587.50,
    2591.75, 2594.50, 2598.75, 2601.25, 2603.25, 2606.00, 2608.75, 2614.50,
    2620.25, 2624.75, 2627.50, 2629.75, 2631.75, 2633.75, 2636.25, 2639.50,
    2640.00, 2642.50, 2646.00, 2655.50, 2660.00, 2664.50, 2666.50, 2669.75,
    2672.25, 2675.00, 2677.00, 2679.00, 2681.50, 2683.75, 2686.00, 2689.00,
    2692.50, 2697.75, 2701.50, 2704.00, 2706.00, 2709.50, 2713.00, 2715.00,
    2717.75, 2720.50, 2726.25, 2728.25, 2730.25, 2735.00, 2738.25, 2741.00,
    2743.25, 2745.25, 2750.75, 2753.50, 2755.50, 2758.00, 2760.00, 2762.50,
    2766.00, 2771.50, 2774.75, 2776.75, 2778.75, 2782.00, 2784.00, 2786.75,
    2789.75, 2793.50, 2796.00, 2798.25, 2800.25, 2804.00, 2806.00, 2810.25,
    2812.50, 2814.50, 2820.00, 2822.50, 2825.00, 2827.25, 2829.25, 2831.25,
    2834.00, 2836.00, 2838.00, 2841.50, 2843.50, 2846.75]

print(f"{len(ACCEPTANCE_FROZEN_TIMES)} frozen / {len(ACCEPTANCE_CORRECTED_TIMES)} corrected reference times")

### 1. The chain, end to end, on synthetic data

Everything except the `.mff` reader: both filters, the trailing box, the
expanding normaliser, all 210 features, both boosters, stage 7. The input is
deterministic noise, so the expected values below fingerprint the whole chain.

This is the check a fresh clone can run with only `models/` and `constants.py`.

In [1]:
"""Synthetic end-to-end check.  ~40 s."""
# Test parameters, not chain constants.  They only have to stay the same
# between runs, which is what makes the fingerprint below comparable.
SYNTH_SEED = 20260906
SYNTH_MINUTES = 3.0
SYNTH_NORM_MINUTES = 1.0
SYNTH_AMPLITUDE_V = 3e-5     # a plausible EEG scale; nothing depends on it

_synthetic_rng = np.random.default_rng(SYNTH_SEED)
_n_samples = int(SYNTH_MINUTES * SECONDS_PER_MINUTE * SAMPLING_RATE_HZ)
_synthetic_signal = _synthetic_rng.standard_normal(
    (len(ANALYSIS_CHANNELS), _n_samples)) * SYNTH_AMPLITUDE_V

_synthetic_detector = StreamingDetector(
    make_array_feed(_synthetic_signal),
    warmup_samples=0,
    n_crop_samples=_n_samples - LOOKAHEAD_SAMPLES,
    normalisation_seconds=SYNTH_NORM_MINUTES * SECONDS_PER_MINUTE,
    detection_threshold=DETECTION_THRESHOLD,
    correction=CORRECTION_BOOSTER, alpha=CORRECTION_ALPHA,
    drop_short_final=False)
_synthetic_detector.fit_normalisation(verbose=False)
_synthetic_result = _synthetic_detector.run(verbose=False)
_scored_proba = _synthetic_result["proba"][_synthetic_result["n_norm_epochs"]:]

_measured = {"n_epochs": int(len(_scored_proba)),
             "proba_sum": float(_scored_proba.sum()),
             "proba_max": float(_scored_proba.max()),
             "n_detections": len(_synthetic_result["fired"])}
_expected = SYNTHETIC_EXPECTED
# counts must match exactly; the float sums get a tolerance because the last
# bits of a sum over 476 probabilities are not portable
SUM_TOLERANCE, MAX_TOLERANCE = 1e-3, 1e-4
_tolerances = {"n_epochs": 0, "n_detections": 0,
               "proba_sum": SUM_TOLERANCE, "proba_max": MAX_TOLERANCE}
if _expected is None:
    print("no fingerprint recorded; this run produced:\n" + json.dumps(_measured, indent=4))
else:
    print(f"{'':14s}{'got':>16s}{'expected':>16s}{'tolerance':>12s}")
    _all_match = True
    for key, expected_value in _expected.items():
        matches = abs(_measured[key] - expected_value) <= _tolerances[key]
        _all_match &= matches
        print(f"{key:14s}{_measured[key]:>16.6g}{expected_value:>16.6g}{_tolerances[key]:>12g}   "
              f"{'ok' if matches else 'DIFFERS'}")
    print("\nPASS - the chain reproduces its fingerprint" if _all_match else
          "\nFAIL - something in the chain differs from what was shipped.\n"
          "       Check the version banner at the top first; mne_features feature\n"
          "       definitions move between releases, and that shows up here.")

                           got        expected   tolerance
n_epochs                   476             476           0   ok
proba_sum              196.516         196.516       0.001   ok
proba_max             0.917924        0.917924      0.0001   ok
n_detections                19              19           0   ok

PASS - the chain reproduces its fingerprint

### 2. The acceptance test: 291 detections on a real recording

The number this notebook was built against. The AO0 recording, minutes 4 to 47.5, frozen
classifier alone: **291 detections**; with the correction: **267**. The times
must be identical, not just the count.

That recording is patient data and is not in this repository, so this cell skips
itself if `ACCEPTANCE_MFF` is missing. To run it, set `ACCEPTANCE_MFF` in
the cell below: it is **not** in the Parameters cell. It is kept as the provenance record.

In [1]:
"""The acceptance test.  Skips cleanly when the recording is absent.  ~2 min."""
# patient data, not in this repository.  CHANGE THIS to where your copy of the
# AO0 recording's .mff directory is.  It is NOT in the Parameters cell.
ACCEPTANCE_MFF = os.path.join(REPO_ROOT, os.pardir, "mff's", "AO0-mff")
ACCEPT_START_MINUTE, ACCEPT_END_MINUTE = 4.0, 47.5   # the lab's trim window
ACCEPT_N_FROZEN = 291     # what the reference package emitted
ACCEPT_N_CORRECTED = 267

if not os.path.isdir(ACCEPTANCE_MFF):
    print(f"SKIPPED - {ACCEPTANCE_MFF} is not present.\n"
          f"          This is expected: the recording is patient data and is "
          f"not distributed.\n"
          f"          Verification 1 above covers the whole chain without it.")
else:
    for _label, _correction, _expected_count, _expected_times in (
            ("frozen only", None, ACCEPT_N_FROZEN, ACCEPTANCE_FROZEN_TIMES),
            ("+ correction", CORRECTION_BOOSTER, ACCEPT_N_CORRECTED,
             ACCEPTANCE_CORRECTED_TIMES)):
        _acceptance_detector = StreamingDetector(
            make_mff_feed(ACCEPTANCE_MFF,
                          ACCEPT_START_MINUTE - FILTER_WARMUP_MINUTES),
            warmup_samples=int(round(FILTER_WARMUP_MINUTES * SECONDS_PER_MINUTE * SAMPLING_RATE_HZ)),
            n_crop_samples=int(round((ACCEPT_END_MINUTE - ACCEPT_START_MINUTE)
                                     * SECONDS_PER_MINUTE * SAMPLING_RATE_HZ)) + 1,
            normalisation_seconds=NORMALISATION_SECONDS,
            detection_threshold=DETECTION_THRESHOLD, correction=_correction,
            alpha=CORRECTION_ALPHA if _correction is not None else 0.0,
            drop_short_final=False)
        _acceptance_detector.fit_normalisation(verbose=False)
        _acceptance_result = _acceptance_detector.run(verbose=False)
        _detection_times = [
            round(detection_time + ACCEPT_START_MINUTE * SECONDS_PER_MINUTE,
                  TIME_DECIMALS)
            for detection_time, _ in _acceptance_result["fired"]]
        _times_match = _detection_times == _expected_times
        print(f"{_label:14s} {len(_detection_times):4d} detections "
              f"(expected {_expected_count})   times "
              f"{'IDENTICAL' if _times_match else 'DIFFER'}")
        if not _times_match:
            SHOW_MISMATCHES = 10
            _unexpected_times = [time_sec for time_sec in _detection_times
                                 if time_sec not in _expected_times]
            _missing_times = [time_sec for time_sec in _expected_times
                              if time_sec not in _detection_times]
            print(f"               extra {len(_unexpected_times)}: "
                  f"{_unexpected_times[:SHOW_MISMATCHES]}")
            print(f"               missing {len(_missing_times)}: "
                  f"{_missing_times[:SHOW_MISMATCHES]}")

frozen only     291 detections (expected 291)   times IDENTICAL
+ correction    267 detections (expected 267)   times IDENTICAL

### 3. The feature definitions are the original ones

`extract_epoch_features` is a faster assembly of the offline notebook's
`extract_epochs_features_mne`. The original is reproduced verbatim in the next
cell, off the runtime path, so the equivalence can be checked.

In [ ]:
# ---------------------------------------------------------------------
# Verbatim from cell 15 of `spm_extraction_annotated.ipynb`, the offline
# notebook the frozen classifier was trained in.  Not on the runtime path.
# Do not tidy it: these definitions are what the model was trained on.
# ---------------------------------------------------------------------
# Feature extraction functions
#
# This block defines the functions that are actually used in the SPM pipeline
# (unlike most of the previous "Core" cell):
#   - extract_epochs_features_mne: computes a large set of features (statistics,
#     entropy, spectral power, etc.) for an array of 250 ms epochs of one channel;
#   - raw_chan_to_feat: takes a "raw" mne.Raw and a channel name, cuts the channel
#     into 250 ms epochs, normalizes it, and calls extract_epochs_features_mne.
# raw_chan_to_feat is what gets called in stage 8 (batch feature extraction),
# and the DataFrames it produces are read by the SPM-calculation function from stage 7.
from mne_features.univariate import get_univariate_funcs, compute_pow_freq_bands
from mne_features.feature_extraction import extract_features
import mne_features.univariate

window_size = 250  # length of one epoch in samples (at 1000 Hz that's 250 ms)


def extract_epochs_features_mne(epochs, subj, sr):
    """
    Computes a set of features (time-domain, spectral, entropy, wavelet, etc.)
    for each epoch in `epochs`, using functions from the mne_features library.
    epochs: array of shape (n_epochs, n_samples_per_epoch)
    subj:   subject identifier (written into the 'subj' column of the result)
    sr:     signal sampling rate, Hz
    Returns a DataFrame: one row per epoch, columns are features.
    """
    data = np.array(epochs)

    # --- basic time-domain and spectral features computed on the whole array of epochs at once ---
    ptp_amp = mne_features.univariate.compute_ptp_amp(data)
    hjorth_mobility = mne_features.univariate.compute_hjorth_mobility(data)
    hjorth_complexity = mne_features.univariate.compute_hjorth_complexity(data)
    samp_entropy = mne_features.univariate.compute_samp_entropy(data, emb=2, metric='chebyshev')
    compute_spect_slope = mne_features.univariate.compute_spect_slope(
        sr, data, fmin=0.1, fmax=50, with_intercept=True, psd_method='welch', psd_params=None
    )

    # Compute power in specified frequency bands with normalization enabled and no ratios
    # frequency bands (Hz) used for computing the signal's power/energy
    bands_dict = {'theta': (4, 8), 'alpha': (8, 12), 'sigma': (12, 16), 'beta': (16, 30), 'gamma': (30, 100),
                  'fast': (100, 300)}

    energy_freq_bands = mne_features.univariate.compute_energy_freq_bands(sr, data, freq_bands=bands_dict)

    app_entropy = mne_features.univariate.compute_app_entropy(data, emb=2, metric='chebyshev')

    decorr_time = mne_features.univariate.compute_decorr_time(sr, data)

    higuchi_fd = mne_features.univariate.compute_higuchi_fd(data, kmax=10)

    hjorth_complexity_spect = mne_features.univariate.compute_hjorth_complexity_spect(sr, data, normalize=False,
                                                                                      psd_method='welch',
                                                                                      psd_params=None)

    hjorth_mobility_spect = mne_features.univariate.compute_hjorth_mobility_spect(sr, data, normalize=False,
                                                                                  psd_method='welch', psd_params=None)

    hurst_exp = mne_features.univariate.compute_hurst_exp(data)

    katz_fd = mne_features.univariate.compute_katz_fd(data)

    kurtosis = mne_features.univariate.compute_kurtosis(data)

    line_length = mne_features.univariate.compute_line_length(data)

    mean = mne_features.univariate.compute_mean(data)

    quantile = mne_features.univariate.compute_quantile(data, q=0.75)

    rms = mne_features.univariate.compute_rms(data)

    skewness = mne_features.univariate.compute_skewness(data)

    spect_entropy = mne_features.univariate.compute_spect_entropy(sr, data, psd_method='welch', psd_params=None)

    std = mne_features.univariate.compute_std(data)

    svd_entropy = mne_features.univariate.compute_svd_entropy(data, tau=2, emb=10)

    svd_fisher_info = mne_features.univariate.compute_svd_fisher_info(data, tau=2, emb=10)

    variance = mne_features.univariate.compute_variance(data)

    zero_crossings = mne_features.univariate.compute_zero_crossings(data, threshold=2.220446049250313e-16)

    abspow = mne_features.univariate.compute_pow_freq_bands(sr, data, {'total': (0.1, 500)}, False,
                                                            psd_method='multitaper')

    # Stack the computed feature arrays into columns for easier DataFrame creation
    # -- collect all the "simple" (one number per epoch) features into a single DataFrame
    stacked_arrays = np.column_stack(
        (hjorth_mobility, hjorth_complexity, samp_entropy, ptp_amp, app_entropy, decorr_time, higuchi_fd,
         hjorth_complexity_spect, hjorth_mobility_spect, hurst_exp, katz_fd, kurtosis, line_length, mean,
         quantile, rms, skewness, spect_entropy, std, svd_entropy, svd_fisher_info,
         variance, zero_crossings, abspow))

    df = pd.DataFrame(stacked_arrays)

    df.columns = ['hjorth_mobility', 'hjorth_complexity', 'samp_entropy', 'ptp_amp', 'app_entropy', 'decorr_time',
                  'higuchi_fd', 'hjorth_complexity_spect', 'hjorth_mobility_spect', 'hurst_exp', 'katz_fd', 'kurtosis',
                  'line_length', 'mean', 'quantile', 'rms', 'skewness', 'spect_entropy',
                  'std', 'svd_entropy', 'svd_fisher_info', 'variance', 'zero_crossings', 'abspow_']

    # Add metadata columns for subject, channel name, epoch label, and epoch ID
    df['subj'] = np.full(df.shape[0], subj)  # `subj` should be defined or passed into the function
    df['epoch_id'] = list(range(0, len(epochs)))  # Assigns a unique ID for each epoch

    # --- features that come back "flattened" and need reshaping into separate columns ---
    kaiser = mne_features.univariate.compute_teager_kaiser_energy(data)
    kaiser_df = pd.DataFrame(
        np.array(kaiser).reshape(-1, 12),
        columns=['teager_kaiser_energy_0_mean', 'teager_kaiser_energy_0_std',
                 'teager_kaiser_energy_1_mean', 'teager_kaiser_energy_1_std',
                 'teager_kaiser_energy_2_mean', 'teager_kaiser_energy_2_std',
                 'teager_kaiser_energy_3_mean', 'teager_kaiser_energy_3_std',
                 'teager_kaiser_energy_4_mean', 'teager_kaiser_energy_4_std',
                 'teager_kaiser_energy_5_mean', 'teager_kaiser_energy_5_std']
    )

    pow_freq_bands = mne_features.univariate.compute_pow_freq_bands(data=epochs, sfreq=sr,
                                                                    freq_bands=bands_dict,
                                                                    normalize=True, ratios=None,
                                                                    ratios_triu=False,
                                                                    psd_method='multitaper', log=False,
                                                                    psd_params=None)
    pow_freq_bands_df = pd.DataFrame(np.array(pow_freq_bands).reshape(-1, 6),
                                     columns=['pow_freq_bands_theta', 'pow_freq_bands_alpha', 'pow_freq_bands_sigma',
                                              'pow_freq_bands_beta', 'pow_freq_bands_gamma', 'pow_freq_bands_fast'])

    # add all pairwise power ratios between bands (theta/alpha, alpha/theta, ...)
    for band1 in ['theta', 'alpha', 'sigma', 'beta', 'gamma', 'fast']:
        for band2 in ['theta', 'alpha', 'sigma', 'beta', 'gamma', 'fast']:
            if band1 == band2:
                continue
            else:
                pow_freq_bands_df[f'pow_freq_bands_{band1}/{band2}'] = pow_freq_bands_df[f'pow_freq_bands_{band1}'] / \
                                                                       pow_freq_bands_df[f'pow_freq_bands_{band2}']

    # Create DataFrame for spectral slope components (intercept, slope, MSE, and R2)
    spect_slope = pd.DataFrame(
        np.array(compute_spect_slope).reshape(-1, 4),
        columns=['spect_slope_intercept', 'spect_slope_slope', 'spect_slope_MSE', 'spect_slope_R2']
    )

    # Create DataFrame for energy frequency bands with each band as a separate column
    energy_freq_bands = pd.DataFrame(
        np.array(energy_freq_bands).reshape(-1, 6),
        columns=['energy_freq_bands_theta', 'energy_freq_bands_alpha', 'energy_freq_bands_sigma',
                 'energy_freq_bands_beta', 'energy_freq_bands_gamma', 'energy_freq_bands_fast']
    )
    # add ratios between bands
    energy_freq_bands['energy_freq_bands_gf'] = energy_freq_bands['energy_freq_bands_gamma'] / energy_freq_bands[
        'energy_freq_bands_fast']
    energy_freq_bands['energy_freq_bands_bg'] = energy_freq_bands['energy_freq_bands_beta'] / energy_freq_bands[
        'energy_freq_bands_gamma']

    wavelet_coef_energy = mne_features.univariate.compute_wavelet_coef_energy(data, wavelet_name='db4')
    wavelet_coef_energy_df = pd.DataFrame(
        np.array(wavelet_coef_energy).reshape(-1, 5),
        columns=['wavelet_coef_energy_0', 'wavelet_coef_energy_1', 'wavelet_coef_energy_2', 'wavelet_coef_energy_3',
                 'wavelet_coef_energy_4'])

    # Concatenate all feature DataFrames (df, spect_slope, pow_freq_bands, energy_freq_bands, kaiser_df) side by side
    subj_features = pd.concat(
        [df, spect_slope, energy_freq_bands, kaiser_df, pow_freq_bands_df, wavelet_coef_energy_df], axis=1)

    # add ratios between bands
    # (second set of ratios - between energy_freq_bands, with short column names using the first letter of the band)
    for band1 in ['theta', 'alpha', 'sigma', 'beta', 'gamma', 'fast']:
        for band2 in ['theta', 'alpha', 'sigma', 'beta', 'gamma', 'fast']:
            if band1 == band2:
                continue
            else:
                subj_features[f'energy_freq_bands_{band1[0]}{band2[0]}'] = subj_features[f'energy_freq_bands_{band1}'] / \
                                                                           subj_features[f'energy_freq_bands_{band2}']
    # Fix the final column order/set - this exact set of features (and in this
    # order) is what the trained model expects (see stage 5/7, model.get_booster().feature_names)
    subj_features = subj_features[
        ['subj', 'epoch_id', 'app_entropy', 'decorr_time', 'energy_freq_bands_theta', 'energy_freq_bands_alpha',
         'energy_freq_bands_sigma', 'energy_freq_bands_beta', 'energy_freq_bands_gamma', 'energy_freq_bands_fast',
         'higuchi_fd', 'hjorth_complexity', 'hjorth_complexity_spect', 'hjorth_mobility', 'hjorth_mobility_spect',
         'hurst_exp', 'katz_fd', 'kurtosis', 'line_length', 'mean', 'pow_freq_bands_theta', 'pow_freq_bands_alpha',
         'pow_freq_bands_sigma', 'pow_freq_bands_beta', 'pow_freq_bands_gamma', 'pow_freq_bands_fast',
         'pow_freq_bands_theta/alpha', 'pow_freq_bands_theta/sigma', 'pow_freq_bands_theta/beta',
         'pow_freq_bands_theta/gamma', 'pow_freq_bands_theta/fast', 'pow_freq_bands_alpha/theta',
         'pow_freq_bands_alpha/sigma', 'pow_freq_bands_alpha/beta', 'pow_freq_bands_alpha/gamma',
         'pow_freq_bands_alpha/fast', 'pow_freq_bands_sigma/theta', 'pow_freq_bands_sigma/alpha',
         'pow_freq_bands_sigma/beta', 'pow_freq_bands_sigma/gamma', 'pow_freq_bands_sigma/fast',
         'pow_freq_bands_beta/theta', 'pow_freq_bands_beta/alpha', 'pow_freq_bands_beta/sigma',
         'pow_freq_bands_beta/gamma', 'pow_freq_bands_beta/fast', 'pow_freq_bands_gamma/theta',
         'pow_freq_bands_gamma/alpha', 'pow_freq_bands_gamma/sigma', 'pow_freq_bands_gamma/beta',
         'pow_freq_bands_gamma/fast', 'pow_freq_bands_fast/theta', 'pow_freq_bands_fast/alpha',
         'pow_freq_bands_fast/sigma', 'pow_freq_bands_fast/beta', 'pow_freq_bands_fast/gamma', 'ptp_amp', 'quantile',
         'rms', 'samp_entropy', 'skewness', 'spect_entropy', 'spect_slope_intercept', 'spect_slope_slope',
         'spect_slope_MSE', 'spect_slope_R2', 'std', 'svd_entropy', 'svd_fisher_info', 'teager_kaiser_energy_0_mean',
         'teager_kaiser_energy_0_std', 'teager_kaiser_energy_1_mean', 'teager_kaiser_energy_1_std',
         'teager_kaiser_energy_2_mean', 'teager_kaiser_energy_2_std', 'teager_kaiser_energy_3_mean',
         'teager_kaiser_energy_3_std', 'teager_kaiser_energy_4_mean', 'teager_kaiser_energy_4_std',
         'teager_kaiser_energy_5_mean', 'teager_kaiser_energy_5_std', 'variance', 'wavelet_coef_energy_0',
         'wavelet_coef_energy_1', 'wavelet_coef_energy_2', 'wavelet_coef_energy_3', 'wavelet_coef_energy_4',
         'zero_crossings', 'abspow_', 'energy_freq_bands_ab', 'energy_freq_bands_ag', 'energy_freq_bands_as',
         'energy_freq_bands_af', 'energy_freq_bands_at', 'energy_freq_bands_bt', 'energy_freq_bands_bs',
         'energy_freq_bands_bg', 'energy_freq_bands_bf', 'energy_freq_bands_st', 'energy_freq_bands_sg',
         'energy_freq_bands_sf', 'energy_freq_bands_gt', 'energy_freq_bands_gf', 'energy_freq_bands_ft']]
    feat = subj_features
    return feat


def raw_chan_to_feat(raw, chan, subj):
    """
    Prepares features for one channel of one subject:
    1) pulls channel `chan` out of `raw` and normalizes it (z-score);
    2) cuts it into epochs of window_size=250 samples (250 ms at 1000 Hz),
       skipping epochs that contain NaN (e.g. from annotations/bad segments);
    3) computes features across the epochs via extract_epochs_features_mne;
    4) adds a few simple statistics over the whole channel
       (chan_ptp/chan_skew/chan_kurt) as extra columns (same value for every epoch).
    Returns a DataFrame with one row per epoch, or an empty DataFrame if
    channel `chan` is not present in `raw`.
    """
    epochs = []
    if chan not in raw.ch_names:
        return pd.DataFrame()
    chan_raw = raw.copy().pick([chan]).get_data(reject_by_annotation='nan').flatten()
    # normalize chan -- bring the channel to zero mean and unit std (z-score)
    chan_norm = (chan_raw - np.nanmean(chan_raw)) / np.nanstd(chan_raw)
    # run on all 250ms epochs (exclude last second)
    for i in range(0, len(chan_norm) - 4 * window_size, window_size):
        if not np.isnan(chan_norm[i: i + window_size]).any():
            epochs.append(chan_norm[i: i + window_size])
    epochs = np.array(epochs)

    curr_feat = extract_epochs_features_mne(epochs, subj, raw.info['sfreq'])
    # Flatten the nested list using list comprehension
    clean_chan = [item for sublist in epochs for item in sublist]
    chan_feat = {
        'chan_name': chan,
        'chan_ptp': np.ptp(clean_chan),
        'chan_skew': sp_stats.skew(clean_chan),
        'chan_kurt': sp_stats.kurtosis(clean_chan),
    }

    for feat in chan_feat.keys():
        curr_feat[feat] = chan_feat[feat]

    return curr_feat

In [1]:
"""The fast extractor equals the notebook's, column for column."""
FEATURE_CHECK_SEED, FEATURE_CHECK_N_EPOCHS = 1, 8   # arbitrary, only repeatable
_random_epochs = np.random.default_rng(FEATURE_CHECK_SEED).standard_normal(
    (FEATURE_CHECK_N_EPOCHS, EPOCH_SAMPLES))
_fast_features = extract_epoch_features(_random_epochs, "chk", SAMPLING_RATE_HZ)
_notebook_features = extract_epochs_features_mne(_random_epochs, "chk", SAMPLING_RATE_HZ)
assert list(_fast_features.columns) == list(_notebook_features.columns), "column order differs"
_numeric_columns = [column for column in _fast_features.columns
                    if column not in ("subj",)]
_max_difference = float(np.abs(
    _fast_features[_numeric_columns].to_numpy(float)
    - _notebook_features[_numeric_columns].to_numpy(float)).max())
print(f"{len(_numeric_columns)} numeric columns x {len(_random_epochs)} epochs, "
      f"max |fast - notebook| = {_max_difference:.3g}")
print("PASS" if _max_difference == 0.0 else "DIFFERS - investigate before trusting the run")

103 numeric columns x 8 epochs, max |fast - notebook| = 0
PASS

---
### What these checks proved when the notebook was built

| check | result |
|---|---|
| synthetic end-to-end fingerprint | reproduced |
| acceptance test, frozen only | 291 detections, times identical to the reference |
| acceptance test, + correction | 267 detections, times identical to the reference |
| fast extractor vs notebook cell 15 | max abs difference 0.0 |

The reference is the research package this notebook was collapsed from
(`detect.py`), run on the same recording and span.